# 02 — `ruff` et `mypy`

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- configurer `ruff` pour le linting et le formatage
- configurer `mypy` en mode strict
- comprendre les principales catégories de règles ruff
- intégrer dans un workflow CI minimal

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- `pyproject.toml`, `uv`
- type hints modernes

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- CI/CD avancé (hors périmètre Intermédiaire)

## Plan

1. `ruff` : linter + formatter ultra-rapide
2. Configuration dans `pyproject.toml`
3. Commandes essentielles
4. `mypy` : vérificateur de types
5. Configuration stricte
6. CI minimal
7. Synthèse
8. Exercices

---

## 1. `ruff` : linter + formatter ultra-rapide

`ruff` (Astral) remplace `flake8`, `isort`, `pyflakes`, `pylint`, `black` en un seul outil 10 à 100x plus rapide (écrit en Rust).

```bash
uv add --dev ruff
uv run ruff check .          # linting
uv run ruff format .         # formatage
uv run ruff check . --fix    # auto-fix
```

---

## 2. Configuration dans `pyproject.toml`

In [ ]:
config = """
[tool.ruff]
target-version = 'py312'
line-length = 100

[tool.ruff.lint]
select = [
    "E",    # pycodestyle errors
    "W",    # pycodestyle warnings
    "F",    # pyflakes
    "I",    # isort
    "N",    # pep8-naming
    "UP",   # pyupgrade
    "B",    # bugbear
    "SIM",  # simplify
    "RUF",  # ruff-specific
]

[tool.ruff.format]
quote-style = "single"
"""
print(config)


### Catégories principales de règles

- **E/W** : PEP 8 (style)
- **F** : erreurs logiques (variable non utilisée, import manquant)
- **I** : tri des imports
- **UP** : mises à jour vers les syntaxes modernes
- **B** : bug potentiels (mutable default, f-string sans placeholders)
- **SIM** : simplifications
- **RUF** : règles spécifiques à ruff

---

## 3. Commandes essentielles

```bash
ruff check .               # vérifie
ruff check . --fix         # auto-corrige
ruff format .              # formate (comme black)
ruff format . --check      # vérifie le formatage
ruff rule E721             # documentation d'une règle
```

---

## 4. `mypy` : vérificateur de types

`mypy` vérifie statiquement les annotations de type. Il attrape des bugs **avant** l'exécution.

```bash
uv add --dev mypy
uv run mypy src/
uv run mypy --strict src/
```

---

## 5. Configuration stricte

In [ ]:
config = """
[tool.mypy]
python_version = '3.12'
strict = true
warn_return_any = true
warn_unused_configs = true
disallow_untyped_defs = true
"""
print(config)


### Erreurs courantes et solutions

| Erreur | Solution |
|---|---|
| `Function is missing a return type` | Ajouter `-> Type` |
| `Incompatible return value type` | Corriger le type |
| `has no attribute "x"` | Type trop vague (`Any` ou `object`) |
| `Missing type parameters for generic type` | `list[int]` au lieu de `list` |

---

## 6. CI minimal

Un fichier GitHub Actions minimal :

```yaml
# .github/workflows/ci.yml
name: CI
on: [push, pull_request]
jobs:
  check:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: astral-sh/setup-uv@v4
      - run: uv sync --frozen
      - run: uv run ruff check .
      - run: uv run ruff format . --check
      - run: uv run mypy src/
      - run: uv run pytest
```

---

## Synthèse

| Outil | Rôle |
|---|---|
| `ruff check` | Linting (détection de problèmes) |
| `ruff format` | Formatage (style) |
| `ruff check --fix` | Auto-correction |
| `mypy --strict` | Vérification de types |


### Règles à retenir

1. **`ruff` en premier** : rapide, couvre large. Configurer dans `pyproject.toml`.
2. **`mypy --strict` pour le code neuf.** Progressif sur le code legacy.
3. **CI = ruff + mypy + pytest.** Les trois ensemble, à chaque push.
4. **Formatter avant de commiter** : `ruff format` (ou configurer un pre-commit hook).

---

## Exercices

Les exercices sont gradués. Tous utilisent des fonctions typées (PEP 604).

### Exercice 1 — Corriger le code *(facile)*

Voici un code avec 3 erreurs que ruff et mypy attraperaient. Trouvez-les et corrigez.

```python
import os, sys
unused = 42
def f(x):
    return x + '1'
f(5)
```

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Ruff_et_mypy", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
# 1. import sys inutilisé → F401
# 2. unused non utilisé → F841
# 3. x + '1' avec x: int → TypeError à l'exécution, mypy error

import os  # sys retiré

def f(x: int) -> int:
    return x + 1

print(f(5))
```

</details>

### Exercice 2 — Configurer ruff *(moyen)*

Écrire la section `[tool.ruff]` dans `pyproject.toml` pour : line-length 120, target Python 3.12, activer E, F, I, UP, B, SIM. Désactiver E501 (line too long — géré par le formatter).

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Ruff_et_mypy", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
config = '''
[tool.ruff]
target-version = "py312"
line-length = 120

[tool.ruff.lint]
select = ["E", "F", "I", "UP", "B", "SIM"]
ignore = ["E501"]
'''
print(config)
```

</details>

### Exercice 3 — Fichier CI complet *(difficile)*

Écrire un fichier `.github/workflows/ci.yml` qui :

1. Tourne sur `push` et `pull_request`
2. Utilise `uv` pour installer
3. Lance `ruff check`, `ruff format --check`, `mypy --strict src/`, `pytest`
4. Échoue si l'un des quatre échoue.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Ruff_et_mypy", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
yaml_content = '''
name: CI
on: [push, pull_request]
jobs:
  check:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: astral-sh/setup-uv@v4
      - run: uv sync --frozen
      - run: uv run ruff check .
      - run: uv run ruff format . --check
      - run: uv run mypy --strict src/
      - run: uv run pytest
'''
print(yaml_content)
```

</details>

---

## Ressources externes

### Documentation officielle
- [ruff docs](https://docs.astral.sh/ruff/)
- [mypy docs](https://mypy.readthedocs.io/)